# Collate jurisdiction yearly mining summaries

Pulls all `*_yearly.json` jurisdiction timeseries from the AMW media CDN,
joins identity metadata (`country`, `name`, `bbox`, …) from the matching
`*_impacts_unfiltered_dict.json` files, and assembles a GeoDataFrame
(bbox → polygon).

Set `DATA_DATE` to the publish folder you want (see `DATA_UPDATED_AT` in
`constants.py`).

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from urllib.request import urlopen

import geopandas as gpd
import pandas as pd
from shapely.geometry import box


In [ ]:
# User: sets the DATA_DATE and OUT_PATH including suffix (.csv or .parquet)

DATA_DATE = "20260822"  # CDN publish folder
BASE = f"https://media-amw.earthgenome.org/{DATA_DATE}"

# Optional local write. Set to None to skip.
OUT_PATH: Path | None = Path(
    f"../../data/boundaries/mined_areas_by_jurisdiction_{DATA_DATE}.csv"
)

JURISDICTIONS = [
    {
        "type": "national_admin",
        "yearly": "data/boundaries/national_admin/out/national_admin_yearly.json",
        "meta": "data/boundaries/national_admin/out/national_admin_impacts_unfiltered_dict.json",
    },
    {
        "type": "subnational_admin",
        "yearly": "data/boundaries/subnational_admin/out/admin_areas_display_yearly.json",
        "meta": "data/boundaries/subnational_admin/out/admin_areas_display_impacts_unfiltered_dict.json",
    },
    {
        "type": "indigenous_territories",
        "yearly": "data/boundaries/protected_areas_and_indigenous_territories/out/indigenous_territories_yearly.json",
        "meta": "data/boundaries/protected_areas_and_indigenous_territories/out/indigenous_territories_impacts_unfiltered_dict.json",
    },
    {
        "type": "protected_areas",
        "yearly": "data/boundaries/protected_areas_and_indigenous_territories/out/protected_areas_yearly.json",
        "meta": "data/boundaries/protected_areas_and_indigenous_territories/out/protected_areas_impacts_unfiltered_dict.json",
    },
]

In [ ]:
def fetch_json(rel: str):
    url = f"{BASE}/{rel}"
    print(f"GET {url}")
    with urlopen(url, timeout=120) as resp:
        return json.loads(resp.read())


def meta_frame(records: list[dict], jurisdiction_type: str) -> pd.DataFrame:
    """Flatten fat-payload identity fields; drop calculator/illegality nests."""
    rows = []
    for r in records:
        name = r.get("name_field")
        if not name:
            # national_admin uses `country` as the display name
            name = r.get("country")
        rows.append(
            {
                "id": r["id"],
                "type": jurisdiction_type,
                "country": r.get("country"),
                "country_code": r.get("country_code"),
                "name": name,
                "status": r.get("status_field"),
                "bbox": r.get("bbox"),  # [minx, miny, maxx, maxy]
            }
        )
    return pd.DataFrame(rows)


frames = []
for spec in JURISDICTIONS:
    yearly = pd.DataFrame(fetch_json(spec["yearly"]))
    meta = meta_frame(fetch_json(spec["meta"]), spec["type"])
    merged = yearly.merge(meta, on="id", how="left", validate="many_to_one")
    missing = merged["type"].isna().sum()
    if missing:
        print(f"  warning: {missing} yearly rows with no metadata for {spec['type']}")
    frames.append(merged)
    print(f"  {spec['type']}: {len(yearly)} yearly rows, {len(meta)} jurisdictions")

df = pd.concat(frames, ignore_index=True)
df.head()

In [ ]:
def bbox_to_polygon(bbox):
    if bbox is None or (isinstance(bbox, float) and pd.isna(bbox)):
        return None
    minx, miny, maxx, maxy = bbox
    return box(minx, miny, maxx, maxy)


gdf = gpd.GeoDataFrame(
    df.drop(columns=["bbox"]),
    geometry=df["bbox"].map(bbox_to_polygon),
    crs="EPSG:4326",
)

# Stable column order
cols = [
    "id",
    "type",
    "country",
    "country_code",
    "name",
    "status",
    "admin_year",
    "intersected_area_ha",
    "intersected_area_ha_cumulative",
    "geometry",
]
gdf = gdf[cols]

print(gdf.dtypes)
print(f"\n{len(gdf):,} rows | {gdf['id'].nunique():,} jurisdictions | types={sorted(gdf['type'].dropna().unique())}")
print(f"null geometries: {gdf.geometry.isna().sum()}")
gdf.groupby("type").size()

In [ ]:
gdf.sort_values(["type", "country", "name", "admin_year"]).head(20)

In [ ]:
if OUT_PATH is not None:
    OUT_PATH = Path(OUT_PATH)
    OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    if OUT_PATH.suffix == ".parquet":
        gdf.to_parquet(OUT_PATH, index=False)
    elif OUT_PATH.suffix == ".csv":
        gdf.to_csv(OUT_PATH, index=False)
    else:
        gdf.to_file(OUT_PATH)
    print(f"Wrote {OUT_PATH.resolve()} ({OUT_PATH.stat().st_size:,} bytes)")
else:
    print("OUT_PATH is None — skipping write")
